# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [ ]:
%pip install nltk scikit-learn pandas numpy -q

In [1]:
import re, math
from collections import Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('gutenberg', quiet=True)

True

### 1. Recuperar y revisar el corpus

Se usa el corpus de Project Gutenberg (18 documentos) y el preprocesamiento de la práctica anterior:
minúsculas y tokens alfanuméricos. **Todas las técnicas usan los mismos documentos y términos procesados.**

In [2]:
# Preprocesamiento (práctica anterior): minúsculas + tokens alfanuméricos
def preprocess_text(text):
    return re.findall(r'\b\w+\b', text.lower())

file_ids = gutenberg.fileids()

# Texto original (para mostrar fragmentos) y texto procesado (para todas las técnicas)
raw_texts = {fid: gutenberg.raw(fid) for fid in file_ids}
corpus_preprocessed = {fid: preprocess_text(raw_texts[fid]) for fid in file_ids}

def obtener_titulo(fid):
    # La primera línea de los textos de NLTK suele ser "[Título by Autor año]"
    m = re.match(r'\s*\[([^\]]+)\]', raw_texts[fid])
    return m.group(1).strip() if m else fid

titulos = {fid: obtener_titulo(fid) for fid in file_ids}

# Listas alineadas: la posición i es el mismo documento en todas las estructuras
doc_ids = list(file_ids)
docs_tokens = [corpus_preprocessed[fid] for fid in doc_ids]
N = len(doc_ids)

tabla = pd.DataFrame({
    "Identificador": doc_ids,
    "Título": [titulos[f] for f in doc_ids],
    "Términos únicos": [len(set(t)) for t in docs_tokens],
})
display(tabla)

longitudes = [len(t) for t in docs_tokens]
vocab = sorted(set(t for doc in docs_tokens for t in doc))   # orden alfabético definido
print("Número de documentos:", N)
print("Tamaño del vocabulario:", len(vocab))
print(f"Longitud mínima / media / máxima: {min(longitudes):,} / {np.mean(longitudes):,.1f} / {max(longitudes):,}")

,Identificador,Título,Términos únicos
0,austen-emma.txt,Emma by Jane Austen 1816,7256
1,austen-persuasion.txt,Persuasion by Jane Austen 1818,5777
2,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,6336
3,bible-kjv.txt,The King James Bible,12744
4,blake-poems.txt,Poems by William Blake 1789,1512
5,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryan...,3900
6,burgess-busterbrown.txt,The Adventures of Buster Bear by Thornton W. B...,1539
7,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carr...,2571
8,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,8267
9,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton...,7739


Número de documentos: 18
Tamaño del vocabulario: 42099
Longitud mínima / media / máxima: 6,936 / 122,178.8 / 854,046


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [3]:
def calcular_tf(tokens):
    """TF(t,d) = f(t,d): conteo crudo de apariciones del término en el documento."""
    return Counter(tokens)

def calcular_df(docs_tokens):
    """DF(t): número de documentos del corpus donde aparece t (una vez por documento)."""
    df = Counter()
    for tokens in docs_tokens:
        df.update(set(tokens))
    return df

def calcular_idf(df, N):
    """IDF(t) = ln(N / DF(t)) + 1  (fórmula de scikit-learn con smooth_idf=False)."""
    return {t: math.log(N / n) + 1 for t, n in df.items()}

def calcular_tfidf(tf, idf):
    """w_{t,d} = TF(t,d) * IDF(t) para los términos presentes en el documento."""
    return {t: f * idf[t] for t, f in tf.items()}

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [4]:
indice = {t: i for i, t in enumerate(vocab)}
V = len(vocab)

# 2) TF
tf_por_doc = [calcular_tf(tokens) for tokens in docs_tokens]
TF = np.zeros((N, V))
for i, tf in enumerate(tf_por_doc):
    for t, f in tf.items():
        TF[i, indice[t]] = f

# 3) DF y 4) IDF
df = calcular_df(docs_tokens)
idf = calcular_idf(df, N)
idf_vec = np.array([idf[t] for t in vocab])

# 5) Matriz TF-IDF (documentos x términos)
X = TF * idf_vec
print("Dimensiones de X:", X.shape)

# Verificación cruzada con la matriz TF construida con CountVectorizer (tu código anterior)
cv = CountVectorizer(token_pattern=r"\b\w+\b")
tf_cv = cv.fit_transform([" ".join(t) for t in docs_tokens])
assert list(cv.get_feature_names_out()) == vocab
print("Diferencia máxima TF explícito vs CountVectorizer:", np.abs(TF - tf_cv.toarray()).max())

# Ejemplo legible: los 10 términos con mayor peso en el primer documento
tfidf_doc0 = calcular_tfidf(tf_por_doc[0], idf)
print("\nTop-10 pesos TF-IDF en", doc_ids[0])
display(pd.Series(tfidf_doc0).sort_values(ascending=False).head(10))

Dimensiones de X: (18, 42099)
Diferencia máxima TF explícito vs CountVectorizer: 0.0

Top-10 pesos TF-IDF en austen-emma.txt


to      5239.000000
the     5201.000000
and     4896.000000
of      4291.000000
i       3178.000000
a       3129.000000
emma    2765.599259
it      2528.000000
her     2469.000000
was     2398.000000
dtype: float64

In [5]:
# Tu versión anterior (sin +1) vs la versión del taller (con +1)
tf_df = pd.DataFrame(TF, index=doc_ids, columns=vocab)
df_series = (tf_df > 0).sum(axis=0)
idf_sin_uno = np.log(N / df_series)
tfidf_sin_uno = tf_df.multiply(idf_sin_uno, axis=1)

print("Diferencia máxima entre TF-IDF sin +1 y con +1:", np.abs(tfidf_sin_uno.values - X).max())
print("Con el +1, los términos presentes en todos los documentos conservan peso (IDF=1) en lugar de 0.")

Diferencia máxima entre TF-IDF sin +1 y con +1: 64023.0
Con el +1, los términos presentes en todos los documentos conservan peso (IDF=1) en lugar de 0.


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [6]:
vec = TfidfVectorizer(
    analyzer=lambda tokens: tokens,   # usa los tokens ya procesados
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
)
X_sk = vec.fit_transform(docs_tokens)

assert list(vec.get_feature_names_out()) == vocab, "El orden de columnas no coincide"
dif_max = np.abs(X - X_sk.toarray()).max()
print("Máxima diferencia absoluta (explícito vs TfidfVectorizer):", dif_max)
assert dif_max < 1e-9

Máxima diferencia absoluta (explícito vs TfidfVectorizer): 0.0


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

**4 Conslultas**

Cuatro consultas con necesidades de información distintas (el corpus está en inglés, por lo que las consultas también).

| Consulta | Texto | Qué espera encontrar el usuario |
|---|---|---|
| q1 | `white whale captain ship sea` | Textos sobre una expedición marítima y la caza de una ballena. |
| q2 | `king ghost murder prince denmark` | Una obra teatral con intriga de corte, fantasmas y asesinato. |
| q3 | `god lord israel king people` | Textos religiosos o bíblicos con pasajes sobre Dios y el pueblo de Israel. |
| q4 | `marriage ball gentleman sisters estate` | Novelas de costumbres sobre matrimonio, bailes y herencias. |

*(Cambia las consultas si prefieres otras; solo mantén la tabla coherente con lo que defines abajo.)*

In [7]:
consultas = {
    "q1": "white whale captain ship sea",
    "q2": "king ghost murder prince denmark",
    "q3": "god lord israel king people",
    "q4": "marriage ball gentleman sisters estate",
}
consultas_tokens = {k: preprocess_text(v) for k, v in consultas.items()}

# Vector de consulta explícito: vocabulario e IDF del corpus (no se recalcula IDF)
def vector_consulta(tokens):
    q = np.zeros(V)
    for t, f in Counter(tokens).items():
        if t in indice:                # términos fuera del vocabulario se ignoran
            q[indice[t]] = f * idf[t]
    return q

Q = np.array([vector_consulta(t) for t in consultas_tokens.values()])

# Con la biblioteca: transform del vectorizador ya ajustado
Q_sk = vec.transform(list(consultas_tokens.values()))
print("Diferencia máxima vectores de consulta:", np.abs(Q - Q_sk.toarray()).max())

# Términos de cada consulta que existen en el vocabulario
for k, toks in consultas_tokens.items():
    print(k, "->", [t for t in toks if t in indice], "| fuera del vocabulario:", [t for t in toks if t not in indice])

Diferencia máxima vectores de consulta: 0.0
q1 -> ['white', 'whale', 'captain', 'ship', 'sea'] | fuera del vocabulario: []
q2 -> ['king', 'ghost', 'murder', 'prince', 'denmark'] | fuera del vocabulario: []
q3 -> ['god', 'lord', 'israel', 'king', 'people'] | fuera del vocabulario: []
q4 -> ['marriage', 'ball', 'gentleman', 'sisters', 'estate'] | fuera del vocabulario: []


In [8]:
# Similitud coseno explícita mediante producto punto y normas
def coseno(q, d):
    denom = np.linalg.norm(q) * np.linalg.norm(d)
    return float(q @ d / denom) if denom > 0 else 0.0

S = np.array([[coseno(q, d) for d in X] for q in Q])        # (4 consultas, N documentos)
S_sk = cosine_similarity(Q_sk, X_sk)
print("Diferencia máxima coseno explícito vs sklearn:", np.abs(S - S_sk).max())
assert np.allclose(S, S_sk, atol=1e-9)

Diferencia máxima coseno explícito vs sklearn: 2.858824288409778e-15


In [9]:
# Detalle para una combinación consulta-documento
qi, nombre_q = 0, "q1"
di = int(np.argmax(S[qi]))               # documento mejor puntuado para q1
q, d = Q[qi], X[di]
compartidos = np.where((q > 0) & (d > 0))[0]

detalle = pd.DataFrame({
    "Término compartido": [vocab[j] for j in compartidos],
    "q_i": q[compartidos],
    "d_i": d[compartidos],
    "Producto parcial": q[compartidos] * d[compartidos],
})
display(detalle)
print("Consulta:", nombre_q, "| Documento:", doc_ids[di])
print("Producto punto:", q @ d)
print("||q||:", np.linalg.norm(q), "| ||d||:", np.linalg.norm(d))
print("Similitud coseno:", coseno(q, d))

,Término compartido,q_i,d_i,Producto parcial
0,captain,1.693147,557.045422,943.159886
1,sea,1.117783,508.591281,568.494706
2,ship,1.405465,728.030926,1023.222064
3,whale,2.098612,2572.898666,5399.516758
4,white,1.117783,314.097033,351.092335


Consulta: q1 | Documento: melville-moby_dick.txt
Producto punto: 8285.485749568366
||q||: 3.427116997258598 | ||d||: 21541.27139181346
Similitud coseno: 0.11223226081560056


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [10]:
def fragmento(texto, tokens_consulta, ancho=220):
    """Fragmento del texto original alrededor de la primera aparición de un término de la consulta."""
    for t in tokens_consulta:
        m = re.search(r'\b' + re.escape(t) + r'\b', texto, flags=re.IGNORECASE)
        if m:
            ini = max(0, m.start() - ancho // 2)
            return "..." + " ".join(texto[ini:ini + ancho].split()) + "..."
    return " ".join(texto[:ancho].split()) + "..."

pd.set_option("display.max_colwidth", 250)
k_top = min(5, N)

for k, (nombre, toks) in enumerate(consultas_tokens.items()):
    orden = np.argsort(-S[k])[:k_top]
    ranking = pd.DataFrame({
        "Posición": range(1, k_top + 1),
        "Identificador": [doc_ids[i] for i in orden],
        "Título": [titulos[doc_ids[i]] for i in orden],
        "Puntuación": [round(S[k, i], 4) for i in orden],
        "Fragmento": [fragmento(raw_texts[doc_ids[i]], toks) for i in orden],
    })
    print(f"\n=== {nombre}: {consultas[nombre]} ===")
    display(ranking)

# Mismo ranking y mismas puntuaciones con la versión de biblioteca
for k in range(len(consultas)):
    assert list(np.argsort(-S[k])[:k_top]) == list(np.argsort(-S_sk[k])[:k_top])
assert np.allclose(S, S_sk, atol=1e-9)
print("\nVersión explícita y versión con bibliotecas coinciden (tolerancia 1e-9).")


=== q1: white whale captain ship sea ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick by Herman Melville 1851,0.1122,"...rings in his belly. ... One of our harpooneers told me that he caught once a whale in Spitzbergen that was white all over."" --A VOYAGE TO GREENLAND, A.D. 1671 HARRIS COLL. ""Several whales have come in upon this coa..."
1,2,austen-persuasion.txt,Persuasion by Jane Austen 1818,0.0330,"...and mentioned a place; and Anne, after the little pause which followed, added-- ""He is a rear admiral of the white. He was in the Trafalgar action, and has been in the East Indies since; he was stationed there, I belie..."
2,3,whitman-leaves.txt,Leaves of Grass by Walt Whitman 1855,0.0122,"...he large imperious waves, Or some lone bark buoy'd on the dense marine, Where joyous full of faith, spreading white sails, She cleaves the ether mid the sparkle and the foam of day, or under many a star at night, By..."
3,4,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,0.0094,"...ee me!"" But all the animals were asleep, and nobody at all looked at Little Jack Rollaround except an old White Owl; and all she said was, ""Who are you?"" The little boy did not like her, so he blew harder, and th..."
4,5,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton 1914,0.0094,"...erlooked the Mediterranean, was covered by an awning and fenced by little lemon and orange trees. Waiters in white aprons were already laying out on white tables the insignia of an early and elegant lunch; and this seem..."



=== q2: king ghost murder prince denmark ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,shakespeare-hamlet.txt,The Tragedie of Hamlet by William Shakespeare 1599,0.0290,...Centinels. Barnardo. Who's there? Fran. Nay answer me: Stand & vnfold your selfe Bar. Long liue the King Fran. Barnardo? Bar. He Fran. You come most carefully vpon your houre Bar. 'Tis now strook...
1,2,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,0.0152,"...LION Once there was a great big jungle; and in the jungle there was a great big Lion; and the Lion was king of the jungle. Whenever he wanted anything to eat, all he had to do was to come up out of his cave in th..."
2,3,shakespeare-macbeth.txt,The Tragedie of Macbeth by William Shakespeare 1603,0.0128,"...and foule is faire, Houer through the fogge and filthie ayre. Exeunt. Scena Secunda. Alarum within. Enter King Malcome, Donalbaine, Lenox, with attendants, meeting a bleeding Captaine. King. What bloody man is tha..."
3,4,bible-kjv.txt,The King James Bible,0.0094,"...[The King James Bible] The Old Testament of the King James Bible The First Book of Moses: Called Genesis 1:1 In the beginning God created the heaven and the earth. 1:2 And the earth was without form, and void; and..."
4,5,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carroll 1865,0.0073,"...t everything that was said, and went by without noticing her. Then followed the Knave of Hearts, carrying the King's crown on a crimson velvet cushion; and, last of all this grand procession, came THE KING AND QUEEN OF H..."



=== q3: god lord israel king people ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,bible-kjv.txt,The King James Bible,0.1061,"...] The Old Testament of the King James Bible The First Book of Moses: Called Genesis 1:1 In the beginning God created the heaven and the earth. 1:2 And the earth was without form, and void; and darkness was upon the..."
1,2,shakespeare-hamlet.txt,The Tragedie of Hamlet by William Shakespeare 1599,0.0646,"...io? You tremble & look pale: Is not this something more then Fantasie? What thinke you on't? Hor. Before my God, I might not this beleeue Without the sensible and true auouch Of mine owne eyes Mar. Is it not like t..."
2,3,shakespeare-macbeth.txt,The Tragedie of Macbeth by William Shakespeare 1603,0.0279,"...ox. What a haste lookes through his eyes? So should he looke, that seemes to speake things strange Rosse. God saue the King King. Whence cam'st thou, worthy Thane? Rosse. From Fiffe, great King, Where the Norwe..."
3,4,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,0.0231,"...n for the needs of his people, and his heart was so filled with his own greatness that he had no thought for God. One day, this proud king was sitting in his place at church, at vesper service; his courtiers were ab..."
4,5,milton-paradise.txt,Paradise Lost by John Milton 1667,0.0207,"...Rose out of Chaos: or, if Sion hill Delight thee more, and Siloa's brook that flowed Fast by the oracle of God, I thence Invoke thy aid to my adventurous song, That with no middle flight intends to soar Above th' A..."



=== q4: marriage ball gentleman sisters estate ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,0.0077,"...ich his age could receive; and the cheerfulness of the children added a relish to his existence. By a former marriage, Mr. Henry Dashwood had one son: by his present lady, three daughters. The son, a steady respectable..."
1,2,austen-persuasion.txt,Persuasion by Jane Austen 1818,0.0066,"...ady Russell, of steady age and character, and extremely well provided for, should have no thought of a second marriage, needs no apology to the public, which is rather apt to be unreasonably discontented when a woman doe..."
2,3,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,0.0057,"...[The Ball and The Cross by G.K. Chesterton 1909] I. A DISCUSSION SOMEWHAT IN THE AIR The flying ship of Professor Lucifer sang through the skies like a silver arrow; the bleak white steel of it, gleaming in the bleak..."
3,4,edgeworth-parents.txt,"The Parent's Assistant, by Maria Edgeworth",0.0056,"...ne hour for it."" ""So much the worse for you. Come, now, I'm quite fresh again, will you have one game at ball? do."" ""No, I tell you I can't; I'm as tired as if I had been working all day long as hard as a horse...."
4,5,austen-emma.txt,Emma by Jane Austen 1816,0.0048,"...ungest of the two daughters of a most affectionate, indulgent father; and had, in consequence of her sister's marriage, been mistress of his house from a very early period. Her mother had died too long ago for her to ha..."



Versión explícita y versión con bibliotecas coinciden (tolerancia 1e-9).


### Interpretación de resultados

* **Pesos TF-IDF:** Los términos "raros" en el corpus (alto IDF) pero que se repiten mucho en un solo texto (alto TF) son los que mandan en la puntuación, como "whale" o "denmark". Las palabras comunes como "white" casi no aportan peso.
* **Precisión de los rankings:** El modelo acertó perfectamente con lo que buscábamos:
  * **q1 (ballena/mar):** *Moby Dick* gana por goleada en el top 1.
  * **q2 (príncipe/fantasma):** *Hamlet* hace match perfecto.
  * **q3 (dios/israel):** *La Biblia (King James)* lidera cómodamente.
  * **q4 (matrimonio/bailes):** Las novelas de Jane Austen dominan los primeros lugares.
* **Limitaciones observadas:**
  * **Longitud extrema:** Los textos varían demasiado en tamaño (de ~6k a más de 850k tokens). Al usar similitud coseno, el tamaño del documento divide y penaliza el score final, por eso los valores máximos son bajitos (apenas rozan el 0.11).
  * **Cero contexto:** El modelo busca coincidencias literales exactas (lexical matching). Si no ponemos la palabra exacta, no la encuentra porque no entiende de sinónimos ni semántica.